In [1]:
from dotenv import load_dotenv
load_dotenv()




True

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from dotenv import load_dotenv
load_dotenv()




True

In [4]:
from anthropic import Anthropic
client = Anthropic()


In [ ]:
from anthropic import Anthropic
import os

client = Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)



In [6]:
def llm(prompt):
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )
    return message.content[0].text

In [7]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

I'd be happy to help! However, I don't have information about which specific course you're referring to. Could you tell me:

1. **Which course** are you asking about?
2. **Where is it offered** (university, online platform, company, etc.)?
3. **When does it start/is it currently running?**

Once I know these details, I can help you figure out:
- Whether late enrollment is possible
- How to register
- Any catch-up materials you might need

What course are you interested in?


In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [9]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [10]:
answer = llm(prompt)
print(answer)

# Yes, you can join now!

Based on the course information:

- **You can start learning immediately** - You don't need to wait for a confirmation email or formal acceptance. You're welcome to begin right away.

- **You can submit homework while the form is open** - Registration is not strictly required; it's mainly used to gauge interest before the start date.

- **Important if you want a certificate** - If you're interested in receiving a certificate, you'll need to submit your project **while the course is still accepting submissions**. So I'd recommend not waiting too long if that's a goal for you.

Feel free to jump in and start learning!


In [11]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [12]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [13]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [15]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [17]:
search_results = index.search(
    question, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [18]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [19]:
search_results = search(question)


In [20]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [21]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [22]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
'''

In [23]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
 
    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()    

In [26]:
prompt = build_prompt(question, search_results) 

In [27]:
prompt = (build_prompt(question, search_results))


In [28]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [29]:
message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

In [30]:
message.content[0].text

"# Yes, you can join now!\n\nBased on the course information, you can start the LLM Zoomcamp immediately. Here's what you need to know:\n\n## Getting Started\n- You **don't need a confirmation email** to begin\n- You can start learning right away using the available materials\n- No registration check is performed against a list\n\n## To Follow the Course:\n\n1. **Access the materials:**\n   - [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n   - [General Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n   - [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\n2. **Follow this workflow:**\n   - Watch lesson videos\n   - Work through notebooks/code\n   - Read homework instructions\n   - Submit answers through the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) before deadlines\n\n## Important: Certificates\n⚠️ **If you want a certificate**, you must:\n- Submit your project **while submissions a

In [31]:
dir(message)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

In [32]:
message.model_dump()

{'id': 'msg_01RMGe1h7Pa1ieAZXYbwEpM8',
 'container': None,
 'content': [{'citations': None,
   'text': "# Yes, you can join now!\n\nBased on the course information, you can start the LLM Zoomcamp immediately. Here's what you need to know:\n\n## Getting Started\n- You **don't need a confirmation email** to begin\n- You can start learning right away using the available materials\n- No registration check is performed against a list\n\n## To Follow the Course:\n\n1. **Access the materials:**\n   - [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n   - [General Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n   - [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\n2. **Follow this workflow:**\n   - Watch lesson videos\n   - Work through notebooks/code\n   - Read homework instructions\n   - Submit answers through the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) before deadlines\n\n## Important:

In [33]:
message.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=539, output_tokens=337, output_tokens_details=None, server_tool_use=None, service_tier='standard')

In [34]:
input_price = 1 / 1_000_000
output_price = 5 / 1_000_000

cost = (
    message.usage.input_tokens * input_price +
    message.usage.output_tokens * output_price
)

cost

0.0022240000000000003

In [35]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
'''

In [36]:
message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    system=INSTRUCTIONS,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

In [37]:
def llm(instructions, user_prompt, model="claude-haiku-4-5-20251001"):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        system=instructions,
        messages=[
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return message.content[0].text


In [38]:
def rag(query, model="claude-haiku-4-5-20251001"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [39]:
answer = rag("How do I get a certificate?")
print(answer)

# How to Get a Certificate

Based on the course information, here's what you need to do to get a certificate:

## Key Requirements:

1. **Enroll in a live cohort** - You can only get a certificate if you finish the course with a "live" cohort. Self-paced mode does not award certificates.

2. **Pass the Capstone Project** - The main requirement is to successfully complete and pass the Capstone project. 

3. **Complete Peer Reviews** - After submitting your Capstone project, you must peer-review 3 capstones from other participants. You can only do peer reviews while the course is running (after the peer-review list is compiled).

## Important Notes:

- **Homework is optional** - You don't need to complete all homework assignments to get a certificate, though they are recommended for reinforcing concepts.

- **Set your official name** - If you want your actual name on the certificate instead of the randomly assigned name (like "Lucid Elbakyan"), you need to edit your Course Profile and en

In [40]:
answer = rag("What does the fox say?")
print(answer)

I don't know.

That question is not related to the course content provided in the context. The context contains information about the Capstone Project, Module 1 Homework, Module 2 Vector Search, and general course-related questions, but it doesn't contain any information about foxes or what they say.

If you have questions about the course materials, I'd be happy to help!
